<a href="https://colab.research.google.com/github/DataAnalytics808/DASC-522-demo-repository/blob/main/demos/Homework_3_template_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; | &nbsp; [https://colab.research.google.com/github/DataAnalytics808/DASC-522-demo-repository/blob/main/demos/Homework_3_template_v5.ipynb](https://colab.research.google.com/github/DataAnalytics808/DASC-522-demo-repository/blob/main/demos/Homework_3_template_v5.ipynb)

This link is provided in case you are viewing this on GitHub, so that it is easy to open directly in Colab.

When opened in Colab, click "Copy to Drive" in the menu above to copy this file to your Colab account.


## Preamble

In [1]:
import sqlite3

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)
import scipy.stats as stats
import seaborn as sns

import statsmodels.api as sm

# for regression
import statsmodels.formula.api as smf
from sklearn import datasets
from statistics import mean

## Import Database

In [3]:
import os
import sqlite3
import urllib.request

# Download database from GitHub if running in Colab / not present locally
db_filename = "weather_data.db"
if not os.path.exists(db_filename):
    url = f"https://raw.githubusercontent.com/DataAnalytics808/DASC-522-demo-repository/main/data/{db_filename}"
    urllib.request.urlretrieve(url, db_filename)

# Read database
my_connection = sqlite3.connect(db_filename)
my_cursor = my_connection.cursor()

query_result = my_cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(query_result.fetchall())


[('X_table',), ('y_table',)]


## Section 1: Data Loading


A. In a marked-up Python .ipynb file, create code to extract data from two tables (X_table & y_table) in the “weather_data.db” SQL database file. Load them into two dataframes df_X and dy_y.  Be sure to retain the “Record_ID” column in both dataframes – this is needed as the two tables aren’t in order. The tutorial covers this procedure.

X_table

In [6]:
data_X = my_cursor.execute("SELECT * FROM X_table").fetchall()

my_column_names = []

for column in query_result.description:
    my_column_names.append(column[0])

my_column_names

['MONTH',
 'DAY',
 'MONTHDAY',
 'HOUR',
 'IF4PMTO7AM',
 'DEWPOINTTEMPERATURE',
 'SEALEVELPRESSURE',
 'TEMP1',
 'TEMP2',
 'TEMP3',
 'TEMP4',
 'TEMP5',
 'TEMP6',
 'Record_ID']

In [8]:
df_X = pd.DataFrame(data_X, columns=my_column_names)
df_X

,MONTH,DAY,MONTHDAY,HOUR,IF4PMTO7AM,DEWPOINTTEMPERATURE,SEALEVELPRESSURE,TEMP1,TEMP2,TEMP3,TEMP4,TEMP5,TEMP6,Record_ID
0,1,1,1.0,1,1,3,1021.5,8,9,12,13,13,13,1
1,1,1,1.0,2,1,3,1021.7,8,8,9,12,13,13,2
2,1,1,1.0,3,1,3,1021.7,7,8,8,9,12,13,3
3,1,1,1.0,4,1,3,1021.3,7,7,8,8,9,12,4
4,1,1,1.0,5,1,3,1021.0,7,7,7,8,8,9,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42272,12,31,13.0,19,1,9,1017.6,18,16,14,12,11,10,42273
42273,12,31,13.0,20,1,9,1017.6,18,18,16,14,12,11,42274
42274,12,31,13.0,21,1,9,1017.9,18,18,18,16,14,12,42275
42275,12,31,13.0,22,1,9,1018.5,18,18,18,18,16,14,42276


y_table

In [9]:
data_Y = my_cursor.execute("SELECT * FROM Y_table").fetchall()

my_column_names = []

for column in query_result.description:
    my_column_names.append(column[0])

my_column_names

['Record_ID', 'AIRTEMPERATURE']

In [10]:
df_Y = pd.DataFrame(data_Y, columns=my_column_names)
df_Y

,Record_ID,AIRTEMPERATURE
0,17163,-14
1,17162,-14
2,17043,-13
3,17040,-13
4,8705,-13
...,...,...
42272,4835,33
42273,38683,33
42274,38684,33
42275,4071,33


A. Use the df.merge() class method to merge the two dataframes on the “Record_ID” column, into a dataframe called df_new. Other methods such as join() might result in the y values not matching with their corresponding X.

In [12]:
# df_new =
df_new = pd.merge(df_X, df_Y, on="Record_ID")
df_new

,MONTH,DAY,MONTHDAY,HOUR,IF4PMTO7AM,DEWPOINTTEMPERATURE,SEALEVELPRESSURE,TEMP1,TEMP2,TEMP3,TEMP4,TEMP5,TEMP6,Record_ID,AIRTEMPERATURE
0,1,1,1.0,1,1,3,1021.5,8,9,12,13,13,13,1,8
1,1,1,1.0,2,1,3,1021.7,8,8,9,12,13,13,2,7
2,1,1,1.0,3,1,3,1021.7,7,8,8,9,12,13,3,7
3,1,1,1.0,4,1,3,1021.3,7,7,8,8,9,12,4,7
4,1,1,1.0,5,1,3,1021.0,7,7,7,8,8,9,5,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42272,12,31,13.0,19,1,9,1017.6,18,16,14,12,11,10,42273,18
42273,12,31,13.0,20,1,9,1017.6,18,18,16,14,12,11,42274,18
42274,12,31,13.0,21,1,9,1017.9,18,18,18,16,14,12,42275,18
42275,12,31,13.0,22,1,9,1018.5,18,18,18,18,16,14,42276,17


B. Drop the “Record_ID”, “MONTH”, and “DAY” columns. Month and day aren’t needed because they are combined into a decimal MONTHDAY feature. For example, January 1st = 1.0, and January 16th = 1.5.

In [13]:
df_new = df_new.drop(columns=["Record_ID", "MONTH", "DAY"])
df_new

,MONTHDAY,HOUR,IF4PMTO7AM,DEWPOINTTEMPERATURE,SEALEVELPRESSURE,TEMP1,TEMP2,TEMP3,TEMP4,TEMP5,TEMP6,AIRTEMPERATURE
0,1.0,1,1,3,1021.5,8,9,12,13,13,13,8
1,1.0,2,1,3,1021.7,8,8,9,12,13,13,7
2,1.0,3,1,3,1021.7,7,8,8,9,12,13,7
3,1.0,4,1,3,1021.3,7,7,8,8,9,12,7
4,1.0,5,1,3,1021.0,7,7,7,8,8,9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
42272,13.0,19,1,9,1017.6,18,16,14,12,11,10,18
42273,13.0,20,1,9,1017.6,18,18,16,14,12,11,18
42274,13.0,21,1,9,1017.9,18,18,18,16,14,12,18
42275,13.0,22,1,9,1018.5,18,18,18,18,16,14,17


C.	Split df_new into X & y dataframes, where y only contains the “AIRTEMPERATURE” feature.

## Section 2: Data Understanding & Modeling

A. Create a Data Understanding table & two python Data Understanding visualizations such as a scatterplot, histogram, pairplot, or correlation matrix.

B. Full model: Model all variables using ols regression, and display a model summary

C.	BIC selection: Use the BIC analysis code from the “Analysis – AIC & BIC” section of the “4 Week 4A feature…” file to create a set of models.    

i.	Use the “BIC Analysis” code to find the best model using BIC. This takes about 3 minutes if you use Colab. Just FYI for this dataset the "best" AIC model contains all features, which is not interesting since the full model contains all features. The BIC model down-selects a bit.   

ii.	Display a model summary of the best BIC model.

D.	RFE selection: Use the “RFE iteration” code from the “Week 4A feature…” file to find the minimum set of features that achieves score = 0.980, which is equivalent to R2 = 0.980. You’ll need to modify the example code to display extra decimal places to determine this.   

i.	Display a model summary of the best RFE model.   

ii.	In a text block, mention which features do not contribute to improving model performance beyond R2 = 0.980.   



comment on features: blah

## Section 3: Model Evaluation


In a text block, add a ~ 200 word evaluation of your modeling efforts. Be sure to discuss the differences in the variables selected by the different methods, and the tradeoff between model performance & model complexity. Please write in a formal writing style based on the guidance in Appendix B. References are not required.  


Evaluation

##Section 4: Capstone Project Plan

Add detail here regarding your proposed plan for your capstone project. The goal is for it to be beneficial to your current organization, a former organization, or to the DoD in general.   

As a reminder, CUI/PII/PHI topics or datasets are not allowed - your dataset must either be available online, or one that you collect or create.
*   Topic, including the label you will be predicting:
*   Type of problem (regression, binary classification, multiclass classification etc):
*   Will you use any advanced techniques like natural language processing, or using images as a model input? This is not required to be 100% successful:
*   Number of datapoints (300+ is best):
*   Number of features, a.k.a. variables or model inputs:

